# ErrP Personalization — Orchestrator

One notebook to run **every** experiment. All real code lives in the `errp_bci` Python package (upload it as a Kaggle Dataset and attach it). Here you only:

1. **Setup** — install deps and put `errp_bci` on the path.
2. **Select** — choose the experiment and which model(s) to run.
3. **Run** — execute.
4. **Report** — tables + plots.

**Experiments:** `primary`, `validation`, `ablation_classweighting`, `ablation_nofilm`, `ablation_feature`, `ablation_preproc`.

**Datasets attached on Kaggle:** INRIA BCI Challenge (for `primary` + all ablations) and ErrP-Coadaptation (for `validation`).

## 1. Setup

In [ ]:
import sys, os, glob

# Riemannian baselines need pyriemann (not preinstalled on Kaggle).
!pip -q install pyriemann

def _find_pkg():
    """Find the errp_bci package anywhere under /kaggle/input (any nesting),
    or in the local working dir. Validates it via __init__.py."""
    cands = glob.glob('/kaggle/input/**/errp_bci', recursive=True)
    cands += [os.path.join(os.getcwd(), 'errp_bci')]
    for c in cands:
        if os.path.isfile(os.path.join(c, '__init__.py')):
            return os.path.dirname(os.path.abspath(c))
    raise FileNotFoundError('errp_bci not found under /kaggle/input. Attach the '
                            'dataset with the errp_bci/ folder, or set _root manually.')

# If auto-detect ever fails, comment the next line and set _root by hand, e.g.:
# _root = '/kaggle/input/datasets/dipuislam/errp-bci/errp_bci'
_root = _find_pkg()

if _root not in sys.path:
    sys.path.insert(0, _root)
print('errp_bci imported from:', _root)

import errp_bci
from errp_bci import run_experiment, list_experiments, list_methods
from errp_bci.config import Config
print('Available experiments:', list_experiments())
print('Device:', Config.DEVICE)

## 2. Select

In [ ]:
EXPERIMENT = "primary"        # primary | validation | ablation_classweighting |
                              #   ablation_nofilm | ablation_feature | ablation_preproc

# == METHODS: None = run all methods this experiment defines. ==================
#   Or pass a subset (copy-paste from the lists below).
#
#   primary / validation  (all 11):
#     ["Supervised", "Pretrain", "Full-MAML", "MAML-ANIL", "SubjectConditioned",
#      "Reptile", "Prototypical", "Matching", "Riemannian", "CovarianceAlignment", "EEGNet"]
#     ("Pretrain" produces both Pretrain-FT and Pretrain-ZeroShot.)
#
#   ablation_classweighting : ["Full-MAML", "MAML-ANIL"]   (each run weighted + uniform)
#   ablation_nofilm         : ["SubjectConditioned", "SubjectConditioned-NoFiLM"]
#   ablation_feature        : ["Full-MAML", "SubjectConditioned", "EEGNet"]  (PCA-32 vs PCA-128)
#   ablation_preproc        : ["Supervised", "Full-MAML"]   (each run full / filter-only / artifact+baseline)
#
#   For ablations leave METHODS = None to run the full preset.
METHODS    = None
SEEDS      = [42, 123, 456]  # Use [42] for a quick smoke test.
K_SHOTS    = [5, 10, 20]

# Optional overrides (leave as-is unless your Kaggle paths differ):
DATASET_ROOT = None          # e.g. '/kaggle/input/inria-bci-challenge/inria-bci-challenge'
FDR          = False         # True -> add Benjamini-Hochberg corrected p-values to stats

# For `validation` only: if ErrP event auto-detection misfires, force the map, e.g.
# Config.MANUAL_ERRP_EVENT_LABEL_MAP = {33035: 1, 33036: 0}

print('Methods that will run:', list_methods(EXPERIMENT) if METHODS is None else METHODS)

## 3. Run

In [ ]:
results = run_experiment(
    EXPERIMENT,
    methods=METHODS,
    seeds=SEEDS,
    k_shots=K_SHOTS,
    dataset_root=DATASET_ROOT,
    fdr=FDR,
)

## 4. Report

In [ ]:
from errp_bci import reporting

for k in K_SHOTS:
    reporting.print_results_table(results, k, 'balanced_accuracy')
for k in K_SHOTS:
    reporting.print_results_table(results, k, 'auroc')

try:
    reporting.plot_adaptation_curves(results,
        save_path=os.path.join(Config.FIGURES_DIR, f'{EXPERIMENT}_adaptation.png'))
except Exception as e:
    print('adaptation plot skipped:', e)

## 5. Reproduce paper statistics (from saved `Results/`, no GPU needed)

In [ ]:
# Regenerates the paper's tables + the subject-level (N=16) exact Wilcoxon
# with Benjamini-Hochberg FDR for every comparison (vs Supervised, vs EEGNet,
# vs Pretrain-FT/ZeroShot). Reads only the per-subject CSVs -- no torch/data.
from errp_bci import analysis

res = analysis.summary(EXPERIMENT)

# To also write statistical_tests_paper.csv / summary_table.csv / macro_f1.csv:
# analysis.summary(EXPERIMENT, save_dir=f'/kaggle/working/{EXPERIMENT}_stats')
#
# Exact PCA retained-variance (needs the dataset; run once):
# analysis.pca_variance('inria'); analysis.pca_variance('coadaptation')